In [4]:
import torch
from torch import nn
from torch.utils.data import random_split, DataLoader, Subset
from torchvision import datasets
from torchvision.transforms import ToTensor
import torch.nn.functional as F
import numpy as np
import os
import pandas as pd
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from torchvision.transforms import ToTensor, Lambda
import matplotlib.colors as mcolors

from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold


import matplotlib.pyplot as plt
import wandb

import os
import datetime

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(device)

In [ ]:
# Open questions / Notes to myself
# On 09/24 I took out the roudning of numbers < 0.001 to 0. Implement back if you think it should be there. 
# Do I have to startify the data? Maybe based on fractional mass ejected?
# I need to do parameter search: lr, epoch 


In [7]:
#This is in place to fix all variables so I can gauge how effective changes are 
torch.manual_seed(42)
np.random.seed(42)

# Make training deterministic
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


#### **Put data into a database**

In [11]:

class CustomDataset(Dataset):
    def __init__(self, labels, data, transform=None, target_transform=None):
        self.labels = labels
        self.data = data
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.labels)


    def __getitem__(self, idx):
        
        data = self.data[idx]
        label = self.labels[idx]

        data = torch.from_numpy(data).type(torch.float)
        label = torch.tensor(label)

        if self.transform:
            data = self.transform(data)
        if self.target_transform:
            label = self.target_transform(label)
                            
        return data, label

In [12]:
#--- Create the dataset ---# 

# 0:Met[Zsun],1:Age[Gyr],2:Rp[Rsun],3:Vinf[km/s],4:Mass1_i[MSUN],5:Mass2_i[Msun],6:R1[Rsun],7:R2[Rsun],8:Label,9:Mass1_f[MSUN],10:Mass2_f[MSUN],11:Sigma
data = np.load('../../data_splits_splot22f_1008.npz')

X_train = data['X_train']
y_train = data['y_train'][:, 1:]
X_val = data['X_val']
y_val = data['y_val'][:, 1:]
X_test = data['X_test']
y_test = data['y_test'][:, 1:]

# Standard Normalize the data 
train_mean = X_train.mean(axis=0)
train_std = X_train.std(axis=0)

X_train = (X_train - train_mean) / train_std
X_test = (X_test - train_mean) / train_std  # apply train stats
X_val = (X_val - train_mean) / train_std  # apply train stats

# Create separate datasets for train and test
train_dataset = CustomDataset(labels = y_train, data = X_train, transform=None, target_transform = None)
val_dataset = CustomDataset(labels = y_val, data = X_val, transform=None, target_transform = None)
test_dataset = CustomDataset(labels = y_test, data = X_test, transform=None, target_transform = None)

#Take this out for final iteration, it is here to make sure data is shuffled the same each time
g = torch.Generator()
g.manual_seed(42)

train_dataloader = DataLoader(train_dataset, batch_size=512, shuffle=True, generator=g)
val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False, generator=g)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False, generator=g)


for batch_data, (batch_labels) in train_dataloader:

    print("Batch data shape:", batch_data.shape)   # Correct way to check shape
    print("Batch labels shape:", batch_labels.shape)
    print("Labels dtype:", batch_labels.dtype)


Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size([512, 5])
Batch labels shape: torch.Size([512, 3])
Labels dtype: torch.float64
Batch data shape: torch.Size

### **Build Neural Network**

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(5, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Linear(128, 3),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        fractions = F.softmax(logits, dim=-1)  # Convert to probabilities
        return fractions


In [ ]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.train()

    train_loss, median_absolute_error = 0, 0
    y_true = [] # Store labels for balanced accuracy
    y_pred = []  # Store labels for balanced accuracy
    initial_total_massess = []

    for X, y in dataloader:
        X = X.to(device, dtype=torch.float32)
        y = y.to(device, dtype=torch.float32)

        # Compute prediction error
        pred = model(X)

        # Get initial (unnormalized masses)
        mass1i = np.exp((X[:, 3].detach().cpu().numpy() * train_std[3] + train_mean[3]))
        mass2i = np.exp((X[:, 4].detach().cpu().numpy() * train_std[4] + train_mean[4]))

        initial_total_masses = mass1i + mass2i
        initial_total_massess.append(initial_total_masses)
        

        # Calculate overall loss
        main_loss = loss_fn(pred[:,:2], y[:,:2])
        ejec_loss = loss_fn(pred[:,2], y[:,2])
        
        loss = main_loss + cfg.training["auxiliary_weight"] * ejec_loss
        train_loss += loss.item() * X.size(0)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Store predictions and true labels
        y_true.append(y.detach().cpu().numpy())  # Convert tensors to NumPy
        y_pred.append(pred.detach().cpu().numpy())

    train_loss /= size
    
    # stack into (N, 2) arrays
    y_true = np.vstack(y_true)
    y_pred = np.vstack(y_pred)

    # assume y_true and y_pred are torch tensors
    errors = np.abs(y_pred - y_true) 
    median_absolute_error = np.median(errors, axis = 0)

    # Transform true labels 
    initial_total_masses = np.concatenate(initial_total_massess)
    true_mass1 = (y_true[:,0]) * initial_total_masses 
    true_mass2 = (y_true[:,1]) * initial_total_masses
    true_mass_ejec = (y_true[:,2]) * initial_total_masses

    #-- Error metric 2: Absolute Errors in Mass 1 
    predicted_mass1 = y_pred[:,0] * initial_total_masses
    predicted_mass2 = y_pred[:,1] * initial_total_masses
    predicted_mass_ejec = y_pred[:,2] * initial_total_masses

    median_abs_error_m1 = np.median(np.abs(predicted_mass1 - true_mass1)) 
    median_abs_error_m2 = np.median(np.abs(predicted_mass2 - true_mass2)) 
    median_abs_error_mejec = np.median(np.abs(predicted_mass_ejec - true_mass_ejec)) 
    
    #-- Error metric 3: Relative errors for cases where at least one star survives
    median_rel_error_m1 = np.median(np.abs(predicted_mass1[true_mass1 != 0.] - true_mass1[true_mass1 != 0. ])/ true_mass1[true_mass1 != 0.])
    median_rel_error_m2 = np.median(np.abs(predicted_mass2[true_mass2 != 0.] - true_mass2[true_mass2 != 0. ])/ true_mass2[true_mass2 != 0.])
    median_rel_error_m_ejec = np.median(np.abs(predicted_mass_ejec[true_mass_ejec != 0.] - true_mass_ejec[true_mass_ejec != 0. ])/ true_mass_ejec[true_mass_ejec != 0.])

    train_median_abs_errors = [median_abs_error_m1, median_abs_error_m2, median_abs_error_mejec]
    train_median_rel_errors = [median_rel_error_m1, median_rel_error_m2, median_rel_error_m_ejec]

    return train_loss, train_median_abs_errors, train_median_rel_errors

def test(dataloader, model, loss_fn, train_median, train_std):
    size = len(dataloader.dataset)
    model = model.to(device)
    model.eval()
    val_loss, median_absolute_error = 0, 0
    y_true = []
    y_pred = []
    initial_total_massess = []
    X_all = []

    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device, dtype=torch.float32)
            y = y.to(device, dtype=torch.float32)

            pred = model(X)

            # Get initial (unnormalized masses)
            mass1i = np.exp((X[:, 3].detach().cpu().numpy() * train_std[3] + train_mean[3]))
            mass2i = np.exp((X[:, 4].detach().cpu().numpy() * train_std[4] + train_mean[4]))

            initial_total_masses = mass1i + mass2i
            initial_total_massess.append(initial_total_masses)

            # Calculate overall loss
            main_loss = loss_fn(pred[:,:2], y[:,:2])
            ejec_loss = loss_fn(pred[:,2], y[:,2])
            loss = main_loss + cfg.training["auxiliary_weight"] * ejec_loss
            val_loss += loss.item() * X.size(0) 
                
            # Store predictions and true labels
            y_true.append(y.detach().cpu().numpy())  # Convert tensors to NumPy
            y_pred.append(pred.detach().cpu().numpy())

    val_loss /= size

    #-- Error metric 1: Absolute Errors in the mass ratios 
    y_true = np.vstack(y_true) # stack into (N, 2) arrays
    y_pred = np.vstack(y_pred) 

    median_absolute_error = np.median(np.abs(y_pred - y_true) , axis = 0)

    # Transform true labels 
    initial_total_masses = np.concatenate(initial_total_massess)
    true_mass1 = (y_true[:,0]) * initial_total_masses 
    true_mass2 = (y_true[:,1]) * initial_total_masses
    true_mass_ejec = (y_true[:,2]) * initial_total_masses

    #-- Error metric 2: Absolute Errors in Mass 1 
    predicted_mass1 = y_pred[:,0] * initial_total_masses
    predicted_mass2 = y_pred[:,1] * initial_total_masses
    predicted_mass_ejec = y_pred[:,2] * initial_total_masses

    median_abs_error_m1 = np.median(np.abs(predicted_mass1 - true_mass1)) 
    median_abs_error_m2 = np.median(np.abs(predicted_mass2 - true_mass2)) 
    median_abs_error_mejec = np.median(np.abs(predicted_mass_ejec - true_mass_ejec)) 
    
    #-- Error metric 3: Relative errors for cases where at least one star survives
    median_rel_error_m1 = np.median(np.abs(predicted_mass1[true_mass1 != 0.] - true_mass1[true_mass1 != 0. ])/ true_mass1[true_mass1 != 0.])
    median_rel_error_m2 = np.median(np.abs(predicted_mass2[true_mass2 != 0.] - true_mass2[true_mass2 != 0. ])/ true_mass2[true_mass2 != 0.])
    median_rel_error_m_ejec = np.median(np.abs(predicted_mass_ejec[true_mass_ejec != 0.] - true_mass_ejec[true_mass_ejec != 0. ])/ true_mass_ejec[true_mass_ejec != 0.])

    Mtot_final = predicted_mass1 + predicted_mass2

    val_median_abs_errors = [median_abs_error_m1, median_abs_error_m2, median_abs_error_mejec]
    val_median_rel_errors = [median_rel_error_m1, median_rel_error_m2, median_rel_error_m_ejec]

    return val_loss, val_median_abs_errors, val_median_rel_errors, y_pred

In [ ]:
# Get today's date as string
date_str = datetime.datetime.now().strftime("%m%d")

# Safety stop in Jupyter
wandbname = "regression_" + date_str + '_' +input("Name for run: ")

if len(wandbname) == 0:
    raise RuntimeError("Stopped — you didn’t provide name for run.")
    

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="elena-gonzalez-northwestern-university",
    # Set the wandb project where this run will be logged.
    project= "ML_SPH_NN_Regression",
    name= wandbname,
    # Track hyperparameters and run metadata.
    config={
        "model":{
            "inputs": 5,
            "labels": 3, 
            "width": [512, 256, 128],
            "layers": 3,
            "activation_function": "relu",
            "output_activation": "softmax", 
            "softmax_dim": -1,
            "model_type": "mlp",
        }, 
        "training": {
            "epochs": 2000,
            "learning_rate": 0.1, 
            "scheduler": "CosineAnnealingLR",
            "optimizer": "SGD",
            "validation": True,
            "loss function": "L1Loss",
            "loss_type": "weighted_multi_component",
            "main_weight": 1.0,      
            "auxiliary_weight": 0.1, 
        }, 
        "data":{
            
            "dataset": "data_n10k_splot22f.csv",
            "reassigning_merger_masses": True, 
            "regression_labels": "[m1f/mtoti, m2f/mtoti, mejef/mtoti]",
            "round_small_values": False, 
            "round_threshold": False,
            "age_transform": "log10", 
            "v_transform": "log10",
            "rp_trabsform": False, 
            "m1i_transform": "ln(x)", 
            "m2i_transform": "ln(x)", 
            "label_transform": False, 
            "train_size": 0.7, 
            "val_size": 0.15,
            "test_size": 0.15, 
            "stratify": False,
            "random_state": 42,
            "standard_normalization_input": True,
            "standard_normalization_labels": False, 
            "train_batch_size": 512,
            "val_batch_size": 128,
            "test_batch_size": 128,
        }
        
    },
)

run_id = wandb.run.id  
cfg = wandb.config
model_name = f"model_{date_str}_{run_id}"
print(model_name)

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR

# Set Hyper-parameters 
epochs = cfg.training["epochs"]

# Initialize list for storing validation loss for plotting
val_loss_history = []  # Store per-fold validation losses
lrs = []  # Store per-fold validation learning rates

best_val_score = np.inf
best_model_state = None

# Load the NN 
model = NeuralNetwork()
model = model.float()

# Set loss functions, optimizer, and learning rate scheduler
loss_fn = nn.L1Loss() 

optimizer = torch.optim.SGD(model.parameters(), lr=cfg.training["learning_rate"])
scheduler = CosineAnnealingLR(optimizer, T_max=epochs) 

# Begin the training 
for t in range(epochs):
    print(t)
    train_loss, train_median_abs_errors, train_median_rel_errors, train(train_dataloader, model, loss_fn, optimizer)
    val_loss, val_median_abs_errors, val_median_rel_errors, _ = test(val_dataloader, model, loss_fn, train_mean, train_std)

    # Log metrics to wandb.
    
    run.log({"train_loss": train_loss, 
            "train_abs_error_m1": train_median_abs_errors[0], 
            "train_abs_error_m2": train_median_abs_errors[1], 
            "train_abs_error_mejec": train_median_abs_errors[2], 
            "train_rel_error_m1": train_median_rel_errors[0], 
            "train_rel_error_m2": train_median_rel_errors[1], 
            "train_rel_error_mejec": train_median_rel_errors[2],
            "val_loss": val_loss, 
            "val_abs_error_m1": val_median_abs_errors[0], 
            "val_abs_error_m2": val_median_abs_errors[1], 
            "val_abs_error_mejec": val_median_abs_errors[2], 
            "val_rel_error_m1": val_median_rel_errors[0], 
            "val_rel_error_m2": val_median_rel_errors[1], 
            "val_rel_error_mejec": val_median_rel_errors[2], 
            "lr": optimizer.param_groups[0]['lr'],
            "epoch": t})
    
    if val_loss < best_val_score:
        best_val_score = val_loss
        best_model_state = model.state_dict()

    scheduler.step()  # update the learning rate
    lrs.append(optimizer.param_groups[0]['lr'])


print("Done!")

# Now save the best model as well as the data's mean and standard deviation
checkpoint = {
    "model_state_dict": best_model_state,
    "train_mean": train_mean,
    "train_std": train_std,
}
torch.save(checkpoint, model_name)


### Evaluation Metrics

In [ ]:
model = NeuralNetwork()
model_name = model_name
checkpoint = torch.load(model_name, map_location=torch.device('mps'))
model.load_state_dict(checkpoint["model_state_dict"])


test_loss, test_median_abs_error_m1, test_median_abs_error_m2, test_median_abs_error_mejec,test_median_rel_error_m1, test_median_rel_error_m2, test_median_rel_error_m_ejec= test(test_dataloader, model, loss_fn, train_mean, train_std)

print(f"\n Test Results:")
print(f"Overall Test Loss: {test_loss:.4f}")
print(f"Absolute Errors M1 [Msun]: {test_median_abs_error_m1:.4f}")
print(f"Absolute Errors M2 [Msun]: {test_median_abs_error_m2:.4f}")
print(f"Relative Errors M1,f  : {test_median_rel_error_m1:.4f}")
print(f"Relative Errors M2,f: {test_median_rel_error_m2:.4f}")

# run.finish()

In [ ]:
plt.scatter(Mtot_initial, Mtot_final, s = 5)
x = np.linspace(0, int(max(np.max(Mtot_final), np.max(Mtot_initial))), 100)
plt.plot(x, x, color = 'red')
plt.ylabel('Mtot_final')
plt.xlabel('Mtot_initial')

## Distribution of Absolute and Relative Errors ##

In [ ]:

def plot_errors(model, X_train, y_train, X_val, y_val, X_test, y_test, train_mean, train_std,
                feature_idx=(0, 1), fixed_values={}, labels=[], 
                point_labels_train=None, point_labels_val=None, point_labels_test=None):
    """
    Plots regression outputs for a PyTorch model using a 2D slice of a higher-dimensional space.
    Points are color-coded by absolute/relative error and labeled with numbers.
    Shows 4 panels: absolute errors (top) and relative errors (bottom) for both star masses.
    
    Parameters:
    - model: Trained PyTorch model.
    - X_train, y_train: Training dataset.
    - X_val, y_val: Validation dataset.
    - X_test, y_test: Test dataset.
    - train_mean, train_std: normalization stats.
    - feature_idx: Tuple (i, j) specifying which two features to plot.
    - fixed_values: {feature_index: value} for fixing other dimensions.
    - labels: list of strings for axis and titles. Expected: [x_label, y_label, ..., etc].
    - point_labels_train: array of labels/numbers to show for each training point
    - point_labels_val: array of labels/numbers for validation points
    - point_labels_test: array of labels/numbers to show for each test point
    """
    import torch
    import numpy as np
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors

    # Step 1: Normalize fixed values
    fixed_values_norm = {k: (v - train_mean[k]) / train_std[k] for k, v in fixed_values.items()}
    
    train_mask = np.logical_and.reduce([np.isclose(X_train[:, k], v, rtol=0.01) for k, v in fixed_values_norm.items()])
    val_mask = np.logical_and.reduce([np.isclose(X_val[:, k], v, rtol=0.01) for k, v in fixed_values_norm.items()])
    test_mask = np.logical_and.reduce([np.isclose(X_test[:, k], v, rtol=0.01) for k, v in fixed_values_norm.items()])

    if (train_mask.sum() == 0 and val_mask.sum() == 0 and test_mask.sum() == 0):
        return "No data points!"

    X_train_filtered, y_train_filtered = X_train[train_mask], y_train[train_mask]
    X_val_filtered, y_val_filtered = X_val[val_mask], y_val[val_mask]
    X_test_filtered, y_test_filtered = X_test[test_mask], y_test[test_mask]

    # Filter point labels as well
    if point_labels_train is not None:
        point_labels_train_filtered = point_labels_train[train_mask]
    else:
        point_labels_train_filtered = None
        
    if point_labels_val is not None:
        point_labels_val_filtered = point_labels_val[val_mask]
    else:
        point_labels_val_filtered = None
        
    if point_labels_test is not None:
        point_labels_test_filtered = point_labels_test[test_mask]
    else:
        point_labels_test_filtered = None

    # Step 2: Unnormalize data
    X_train_filtered_unnorm = np.array(X_train_filtered) * train_std + train_mean
    X_val_filtered_unnorm = np.array(X_val_filtered) * train_std + train_mean
    X_test_filtered_unnorm = np.array(X_test_filtered) * train_std + train_mean

    y_train_filtered_unnorm = np.array(y_train_filtered) 
    y_val_filtered_unnorm = np.array(y_val_filtered)
    y_test_filtered_unnorm = np.array(y_test_filtered) 

    #Convert masses into not logged 
    X_train_filtered_unnorm[:,3] = np.exp(X_train_filtered_unnorm[:,3])
    X_train_filtered_unnorm[:,4] = np.exp(X_train_filtered_unnorm[:,4])

    # Convert fractions to actual masses
    y_train_filtered_unnorm[:,0] = y_train_filtered_unnorm[:,0] * (X_train_filtered_unnorm[:,3] + X_train_filtered_unnorm[:,4])
    y_train_filtered_unnorm[:,1] = y_train_filtered_unnorm[:,1] * (X_train_filtered_unnorm[:,3] + X_train_filtered_unnorm[:,4])

    y_val_filtered_unnorm[:,0] = y_val_filtered_unnorm[:,0] * (X_val_filtered_unnorm[:,3] + X_val_filtered_unnorm[:,4])
    y_val_filtered_unnorm[:,1] = y_val_filtered_unnorm[:,1] * (X_val_filtered_unnorm[:,3] + X_val_filtered_unnorm[:,4])

    y_test_filtered_unnorm[:,0] = y_test_filtered_unnorm[:,0] * (X_test_filtered_unnorm[:,3] + X_test_filtered_unnorm[:,4])
    y_test_filtered_unnorm[:,1] = y_test_filtered_unnorm[:,1] * (X_test_filtered_unnorm[:,3] + X_test_filtered_unnorm[:,4])

    # Step 3: Get model predictions for the filtered data
    model.eval()
    with torch.no_grad():
        # Predictions for training data
        X_train_tensor = torch.tensor(X_train_filtered, dtype=torch.float32).to(device)
        preds_train = model(X_train_tensor).detach().cpu().numpy()
        
        # Predictions for validation data
        X_val_tensor = torch.tensor(X_val_filtered, dtype=torch.float32).to(device)
        preds_val = model(X_val_tensor).detach().cpu().numpy()
        
        # Predictions for test data  
        X_test_tensor = torch.tensor(X_test_filtered, dtype=torch.float32).to(device)
        preds_test = model(X_test_tensor).detach().cpu().numpy()

    # Convert predicted fractions to actual masses
    preds_train_mass = np.zeros_like(preds_train)
    preds_train_mass[:,0] = preds_train[:,0] * (X_train_filtered_unnorm[:,3] + X_train_filtered_unnorm[:,4])
    preds_train_mass[:,1] = preds_train[:,1] * (X_train_filtered_unnorm[:,3] + X_train_filtered_unnorm[:,4])
    
    preds_val_mass = np.zeros_like(preds_val)
    preds_val_mass[:,0] = preds_val[:,0] * (X_val_filtered_unnorm[:,3] + X_val_filtered_unnorm[:,4])
    preds_val_mass[:,1] = preds_val[:,1] * (X_val_filtered_unnorm[:,3] + X_val_filtered_unnorm[:,4])
    
    preds_test_mass = np.zeros_like(preds_test)
    preds_test_mass[:,0] = preds_test[:,0] * (X_test_filtered_unnorm[:,3] + X_test_filtered_unnorm[:,4])
    preds_test_mass[:,1] = preds_test[:,1] * (X_test_filtered_unnorm[:,3] + X_test_filtered_unnorm[:,4])

    # Step 4: Calculate absolute and relative errors
    abs_error_train = np.abs(preds_train_mass - y_train_filtered_unnorm)
    abs_error_val = np.abs(preds_val_mass - y_val_filtered_unnorm)
    abs_error_test = np.abs(preds_test_mass - y_test_filtered_unnorm)
    
    # Calculate relative errors (avoid division by zero)
    rel_error_train = np.zeros_like(abs_error_train)
    rel_error_val = np.zeros_like(abs_error_val)
    rel_error_test = np.zeros_like(abs_error_test)
    
    # Only calculate relative error where true value > small threshold
    threshold = 0.01  # Solar masses
    
    for i in range(2):  # For both star masses
        train_nonzero = y_train_filtered_unnorm[:, i] > threshold
        val_nonzero = y_val_filtered_unnorm[:, i] > threshold
        test_nonzero = y_test_filtered_unnorm[:, i] > threshold
        
        if train_nonzero.sum() > 0:
            rel_error_train[train_nonzero, i] = abs_error_train[train_nonzero, i] / y_train_filtered_unnorm[train_nonzero, i]
        if val_nonzero.sum() > 0:
            rel_error_val[val_nonzero, i] = abs_error_val[val_nonzero, i] / y_val_filtered_unnorm[val_nonzero, i]
        if test_nonzero.sum() > 0:
            rel_error_test[test_nonzero, i] = abs_error_test[test_nonzero, i] / y_test_filtered_unnorm[test_nonzero, i]

    # Step 5: Create 4-panel plot (2x2 grid)
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))

    # Set color scales
    all_abs_errors = [abs_error_train, abs_error_val, abs_error_test]
    all_rel_errors = [rel_error_train, rel_error_val, rel_error_test]
    
    abs_error_max = max([err.max() for err in all_abs_errors])
    rel_error_max = max([err.max() for err in all_rel_errors])
    
    cmap = plt.cm.Reds  # Use Reds colormap where darker = higher error
    abs_norm = mcolors.Normalize(vmin=0, vmax=abs_error_max)
    rel_norm = mcolors.Normalize(vmin=0, vmax=min(rel_error_max, 5.0))  # Cap relative error display at 500%

    def plot_scatter_with_labels(ax, X_data, error_data, labels, norm, marker, dataset_name, alpha=0.8):
        """Helper function to plot scatter points with labels"""
        if len(X_data) > 0:
            scatter = ax.scatter(X_data[:, feature_idx[0]], X_data[:, feature_idx[1]],
                               c=error_data, cmap=cmap, norm=norm, 
                               marker=marker, s=60, alpha=alpha,
                               edgecolor="k", linewidth=0.5,
                               label=dataset_name)
            
            # Add labels if provided
            if labels is not None:
                for j, label in enumerate(labels):
                    ax.annotate(str(label), 
                               (X_data[j, feature_idx[0]], X_data[j, feature_idx[1]]),
                               xytext=(3, 3), textcoords='offset points',
                               fontsize=7, ha='left', va='bottom',
                               bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
            return scatter
        return None

    # Plot titles and labels
    star_names = ['Star 1 Final Mass', 'Star 2 Final Mass']
    error_types = ['Absolute Error', 'Relative Error']
    
    for i in range(2):  # Star 1 and Star 2
        for j in range(2):  # Absolute and Relative errors
            ax = axes[j, i]
            
            if j == 0:  # Absolute error row
                error_train = abs_error_train[:, i]
                error_val = abs_error_val[:, i]
                error_test = abs_error_test[:, i]
                norm = abs_norm
                unit_label = "(Solar Masses)"
            else:  # Relative error row
                error_train = rel_error_train[:, i]
                error_val = rel_error_val[:, i]
                error_test = rel_error_test[:, i]
                norm = rel_norm
                unit_label = "(Fractional)"
            
            # Plot datasets
            plot_scatter_with_labels(ax, X_train_filtered_unnorm, error_train, 
                                   point_labels_train_filtered, norm, "o", "Train")
            plot_scatter_with_labels(ax, X_val_filtered_unnorm, error_val, 
                                   point_labels_val_filtered, norm, "s", "Val")
            plot_scatter_with_labels(ax, X_test_filtered_unnorm, error_test, 
                                   point_labels_test_filtered, norm, "^", "Test")

            ax.set_xlabel(fr"{labels[0]}" if len(labels) > 0 else "Feature 1", fontsize=12)
            ax.set_ylabel(fr"{labels[1]}" if len(labels) > 1 else "Feature 2", fontsize=12)
            ax.tick_params(axis='both', which='major', labelsize=10)
            ax.set_title(f'{star_names[i]} - {error_types[j]}', fontsize=13)
            ax.legend(fontsize=10)
            ax.grid(True, alpha=0.3)
    
    # Create colorbars
    plt.tight_layout(rect=[0, 0, 0.85, 0.95])  # Leave space for colorbars and title
    
    # Absolute error colorbar (for top row)
    abs_cbar_ax = fig.add_axes([0.87, 0.55, 0.02, 0.35])
    abs_cbar = fig.colorbar(plt.cm.ScalarMappable(norm=abs_norm, cmap=cmap),
                           cax=abs_cbar_ax, orientation='vertical')
    abs_cbar.set_label('Absolute Error\n(Solar Masses)', fontsize=11)
    
    # Relative error colorbar (for bottom row)
    rel_cbar_ax = fig.add_axes([0.87, 0.1, 0.02, 0.35])
    rel_cbar = fig.colorbar(plt.cm.ScalarMappable(norm=rel_norm, cmap=cmap),
                           cax=rel_cbar_ax, orientation='vertical')
    rel_cbar.set_label('Relative Error\n(Fractional)', fontsize=11)

    # Title
    if len(labels) >= 5:
        fig.suptitle(fr"$\mathrm{{M_1}} = {labels[2]}\ M_\odot,\ \mathrm{{M_2}} = {labels[3]}\ M_\odot,\ \mathrm{{Time}} = {round(10**(float(labels[4])),3)}\ \mathrm{{Gyr}}$", 
                     fontsize=16, y=0.98)

    return fig


# Example usage:
# train_labels = np.arange(len(X_train))  
# val_labels = np.arange(len(X_val)) + 1000 
# test_labels = np.arange(len(X_test)) + 2000  

# fig = plot_errors(model, X_train, y_train, X_val, y_val, X_test, y_test, 
#                   train_mean, train_std,
#                   feature_idx=(0, 1), 
#                   fixed_values={2: some_value, 3: m1_value, 4: m2_value},
#                   labels=['Feature 1', 'Feature 2', 'M1_value', 'M2_value', 'time_value'],
#                   point_labels_train=train_labels,
#                   point_labels_val=val_labels,
#                   point_labels_test=test_labels)


In [ ]:
# 0:Age[Gyr],1:Rp,2:Vinf[km/s]3:Mass1[MSUN],4:Mass2[Msun],
from matplotlib.backends.backend_pdf import PdfPages
unique_times = np.unique(x_data_og[:,0])
unique_mass = np.unique(x_data_og[:,4])

for time in unique_times:
    for mass1 in unique_mass:
        for mass2 in unique_mass:
            # print("Time: ", str(time) + " Mass1: " + str(mass1) + " Mass2: " + str(mass2) + '\n' )
            labels = [r'$\rm r_p \, [R_{\odot}]$', r'$\rm Log_{10}(v_{inf} \, [km/s])$', str(mass1), str(round(mass2,2)), str(time)]
            
            fig = plot_errors(model, train_dataset.data, train_dataset.labels, val_dataset.data, val_dataset.labels, test_dataset.data, test_dataset.labels, train_mean, train_std, 
                    feature_idx=(1, 2), fixed_values={0: time, 3: mass1, 4: mass2}, labels = labels, point_labels_train= Train_labels, point_labels_val = Val_labels, point_labels_test=Test_labels )
            if fig != "No data points!":
                # Save the current figure to the PDF
                plt.show()
                plt.close(fig )    # Close it to avoid overlap




In [ ]:


def plot_4d_regression(model, X_train, y_train, X_test, y_test, train_mean, train_std,
                       feature_idx=(0, 1), fixed_values={}, labels=[] ):
    """
    Plots regression outputs for a PyTorch model using a 2D slice of a higher-dimensional space.
    Produces one panel per output dimension (color gradient).
    
    Parameters:
    - model: Trained PyTorch model.
    - X_train, y_train, X_test, y_test: datasets.
    - train_mean, train_std: normalization stats.
    - feature_idx: Tuple (i, j) specifying which two features to plot.
    - fixed_values: {feature_index: value} for fixing other dimensions.
    - labels: list of strings for axis and titles. Expected: [x_label, y_label, ..., etc].
    """

    # Step 1: Normalize fixed values
    fixed_values_norm = {k: (v - train_mean[k]) / train_std[k] for k, v in fixed_values.items()}

    train_mask = np.logical_and.reduce([np.isclose(X_train[:, k], v, rtol=0.01) for k, v in fixed_values_norm.items()])
    test_mask = np.logical_and.reduce([np.isclose(X_test[:, k], v, rtol=0.01) for k, v in fixed_values_norm.items()])

    if (train_mask.sum() == 0 and test_mask.sum() == 0):
        return "No data points!"

    X_train_filtered, y_train_filtered = X_train[train_mask], y_train[train_mask]
    X_test_filtered, y_test_filtered = X_test[test_mask], y_test[test_mask]

    # Step 2: Create meshgrid
    X_train_filtered_unnorm = np.array(X_train_filtered) * train_std + train_mean
    X_test_filtered_unnorm = np.array(X_test_filtered) * train_std + train_mean

    #Convert masses into not logged 
    X_train_mass1 = np.exp(X_train_filtered_unnorm[:,3])
    X_train_mass2 = np.exp(X_train_filtered_unnorm[:,4])

    X_test_mass1 = np.exp(X_test_filtered_unnorm[:,3])
    X_test_mass2 = np.exp(X_test_filtered_unnorm[:,4])

    y_train_filtered_unnorm = np.array(y_train_filtered) 
    y_test_filtered_unnorm = np.array(y_test_filtered) 

    # Correct Units 
    y_train_filtered_unnorm[:,0] = y_train_filtered_unnorm[:,0] * (X_train_mass1 + X_train_mass2)
    y_train_filtered_unnorm[:,1] = y_train_filtered_unnorm[:,1] * (X_train_mass1 + X_train_mass2)

    y_test_filtered_unnorm[:,0] = y_test_filtered_unnorm[:,0] * (X_test_mass1 + X_test_mass2)
    y_test_filtered_unnorm[:,1] = y_test_filtered_unnorm[:,1] * (X_test_mass1 + X_test_mass2)
    
    x_min, x_max = X_train_filtered_unnorm[:, feature_idx[0]].min() - 0.1, X_train_filtered_unnorm[:, feature_idx[0]].max() + 0.1
    y_min, y_max = X_train_filtered_unnorm[:, feature_idx[1]].min() - 0.1, X_train_filtered_unnorm[:, feature_idx[1]].max() + 0.1

    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                         np.linspace(y_min, y_max, 500))

    # Normalize grid
    xx_norm = (xx - train_mean[feature_idx[0]]) / train_std[feature_idx[0]]
    yy_norm = (yy - train_mean[feature_idx[1]]) / train_std[feature_idx[1]]

    # Step 3: Build input space
    X_vis = torch.zeros((xx_norm.ravel().shape[0], X_train.shape[1]), dtype=torch.float32)
    X_vis[:, feature_idx[0]] = torch.tensor(xx_norm.ravel(), dtype=torch.float32)
    X_vis[:, feature_idx[1]] = torch.tensor(yy_norm.ravel(), dtype=torch.float32)
    
    for k, v in fixed_values_norm.items():
        X_vis[:, k] = v

    # Step 4: Model predictions
    X_vis = X_vis.to(device)
    model.eval()
    with torch.no_grad():
        preds = model(X_vis).detach().cpu().numpy()   # shape (Ngrid, 2)

    # Changing the training and testing data to be M1f and M2f 
        
    y_train_filtered_corrected = np.empty((len(y_train_filtered_unnorm[:,0]), 2))
    y_test_filtered_corrected = np.empty((len(y_test_filtered_unnorm[:,0]), 2))
    
    y_train_filtered_corrected[:,0] = y_train_filtered_unnorm[:, 0]
    y_train_filtered_corrected[:,1] = y_train_filtered_unnorm[:, 1] 

    y_test_filtered_corrected[:,0] = y_test_filtered_unnorm[:, 0]
    y_test_filtered_corrected[:,1] = y_test_filtered_unnorm[:, 1] 

    M1_f = preds[:,0] * (np.exp(fixed_values[3]) + np.exp(fixed_values[4]))
    M2_f = preds[:,1] * (np.exp(fixed_values[3]) + np.exp(fixed_values[4]))

    # Reshape into grid for each output dimension
    M1_f = M1_f.reshape(xx.shape)
    M2_f = M2_f.reshape(xx.shape)

    # Step 5: Plot two panels
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    vmin = min(y_train_filtered_corrected[:,0].min(), y_train_filtered_corrected[:,1].min(),
           y_test_filtered_corrected[:,0].min(), y_test_filtered_corrected[:,1].min(),
           M1_f.min(), M2_f.min())
    vmax = max(y_train_filtered_corrected[:,0].max(), y_train_filtered_corrected[:,1].max(),
           y_test_filtered_corrected[:,0].max(), y_test_filtered_corrected[:,1].max(),
           M1_f.max(), M2_f.max())
    
    cmap = plt.cm.coolwarm
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)

    for i, (Z, ax, title) in enumerate(zip([M1_f, M2_f], axes, ['Star 1 Final Mass', 'Star 2 Final Mass'])):
        im = ax.contourf(xx, yy, Z, levels = 100, cmap=cmap, norm = norm)
        scatter1 = ax.scatter(X_train_filtered_unnorm[:, feature_idx[0]],
                              X_train_filtered_unnorm[:, feature_idx[1]],
                              c=y_train_filtered_corrected[:, i], cmap=cmap, norm = norm, edgecolor="k", marker="o", label="Train")
        scatter2 = ax.scatter(X_test_filtered_unnorm[:, feature_idx[0]],
                              X_test_filtered_unnorm[:, feature_idx[1]],
                              c=y_test_filtered_corrected[:, i], cmap=cmap, norm = norm, marker="^", label="Test")

        ax.set_xlabel(fr"{labels[0]}", fontsize=14)
        ax.set_ylabel(fr"{labels[1]}", fontsize=14)
        ax.tick_params(axis='both', which='major', labelsize=12)
        ax.set_title(title, fontsize=15)
    
   
    # After plotting the contours and scatters
    plt.tight_layout(rect=[0,0,0.9,1])  # leave 10% space on the right for the colorbar

    # Create the colorbar on a dedicated axis outside the panels
    cbar_ax = fig.add_axes([0.9999, 0.15, 0.02, 0.7])  # [left, bottom, width, height]
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                    cax=cbar_ax, orientation='vertical', shrink=0.8)
    cbar.set_label('Final Mass', fontsize=14)

    fig.suptitle(fr"$\mathrm{{M_1}} = {labels[2]}\ M_\odot,\ \mathrm{{M_2}} = {labels[3]}\ M_\odot,\ \mathrm{{Time}} = {round(10**(float(labels[4])),3)}\ \mathrm{{Gyr}}$", fontsize = 17)

    plt.tight_layout()
    return fig




In [ ]:
# 0:Age[Gyr],1:Rp,2:Vinf[km/s]3:Mass1[MSUN],4:Mass2[Msun],
from matplotlib.backends.backend_pdf import PdfPages
unique_times = np.unique(x_data_og[:,0])
unique_mass = np.unique(x_data_og[:,4])
# unique_times = [-3]
# unique_mass = [4.0]
with PdfPages('decision_boundary_plots_22f_regression_test_0924.pdf') as pdf:
    for time in unique_times:
        for mass1 in unique_mass:
            for mass2 in unique_mass:
                # print("Time: ", str(time) + " Mass1: " + str(mass1) + " Mass2: " + str(mass2) + '\n' )
                
                labels = [r'$\rm r_p \, [R_{\odot}]$', r'$\rm Log_{10}(v_{inf} \, [km/s])$', str(round(np.exp(mass1),2)), str(round(np.exp(mass2),2)), str(time)]
                
                fig = plot_4d_regression(model, train_dataset.data, train_dataset.labels, test_dataset.data, test_dataset.labels, train_mean, train_std, 
                       feature_idx=(1, 2), fixed_values={0: time, 3: mass1, 4: mass2}, labels = labels)
                if fig != "No data points!":
                    # Save the current figure to the PDF
                    pdf.savefig(fig , bbox_inches='tight')  # Saves the current figure
                    plt.show()
                    plt.close(fig)    # Close it to avoid overlap

In [ ]:
# 0:Age[Gyr],1:Rp,2:Vinf[km/s]3:Mass1[MSUN],4:Mass2[Msun],
from matplotlib.backends.backend_pdf import PdfPages
unique_times = np.unique(x_data_og[:,0])
unique_mass = np.unique(x_data_og[:,4])
for time in unique_times:
    for mass1 in unique_mass:
        for mass2 in unique_mass:
            # print("Time: ", str(time) + " Mass1: " + str(mass1) + " Mass2: " + str(mass2) + '\n' )
            labels = [r'$\rm r_p \, [R_{\odot}]$', r'$\rm Log_{10}(v_{inf} \, [km/s])$', str(mass1), str(round(mass2,2)), str(time)]
            fig = plot_4d_regression(model, train_dataset.data, train_dataset.labels, test_dataset.data, test_dataset.labels, train_mean, train_std,  
                    feature_idx=(1, 2), fixed_values={0: time, 3: mass1, 4: mass2}, labels = labels)
            if fig != "No data points!":
                fig.show()

            print(fig)